In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import os
from time import perf_counter

In [2]:
start = perf_counter()

In [3]:
fake = Faker('en_GB')
random.seed(42)
np.random.seed(42)

# =========================================================
# CONFIG
# =========================================================

NUM_PROPERTIES = 50
START_DATE = '2023-01-01'
END_DATE = '2025-12-31'
OUTPUT_FOLDER = r"C:\Users\geoff\Downloads\Key Docs\7. Airbnb_dbt_snowflake_project\AIRBNB_DBT_PROJECT\source_data"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [4]:
# =========================================================
# DIMENSIONS
# =========================================================

platforms = ['Airbnb', 'Booking.com', 'Vrbo']
property_types = ['Apartment', 'Studio', 'House', 'Loft']
cities = ['London', 'Manchester', 'Birmingham', 'Liverpool']

# =========================================================
# PROPERTY TABLE
# =========================================================

properties = []

for i in range(1, NUM_PROPERTIES + 1):
    properties.append({
        'property_id': f'PROP{i:03}', # all ids 3 numbers long, padded with Zeros
        'property_name': f'{fake.street_name()} {random.choice(property_types)}',
        'city': random.choice(cities),
        'property_type': random.choice(property_types),
        'bedrooms': random.randint(1, 5),
        'max_guests': random.randint(2, 10),
        'owner_name': fake.name(),
        'base_nightly_rate': random.randint(80, 300)
    })

properties_df = pd.DataFrame(properties)
properties_df.to_csv(f'{OUTPUT_FOLDER}/dim_properties.csv', index=False)

In [5]:
# =========================================================
# DATE RANGE
# =========================================================

all_dates = pd.date_range(start=START_DATE, end=END_DATE)

In [6]:
# =========================================================
# SEASONALITY FUNCTIONS
# =========================================================

def get_month_multiplier(month):
    if month in [6, 7, 8]:
        return 1.35
    elif month in [11, 12]:
        return 1.20
    elif month in [1, 2]:
        return 0.85
    else:
        return 1.0

# =========================================================
# GENERATE DAILY PROPERTY TABLE
# =========================================================

property_daily = []
bookings = []
reviews = []

booking_counter = 1
review_counter = 1

for _, prop in properties_df.iterrows():

    booked_dates = set()

    current_date = pd.to_datetime(START_DATE)

    while current_date <= pd.to_datetime(END_DATE):

        dow = current_date.dayofweek # Monday = 0 and Sunday = 6
        month = current_date.month

        # Weekend boost
        weekend_boost = 0.18 if dow in [4, 5] else 0

        # Seasonal occupancy
        seasonality = {
            1: 0.55,
            2: 0.58,
            3: 0.65,
            4: 0.72,
            5: 0.75,
            6: 0.82,
            7: 0.90,
            8: 0.92,
            9: 0.78,
            10: 0.70,
            11: 0.68,
            12: 0.80
        }

        occupancy_probability = seasonality[month] + weekend_boost

        is_booked = np.random.rand() < occupancy_probability

        if current_date in booked_dates:
            current_date += timedelta(days=1)
            continue

        nightly_rate = round(
            prop['base_nightly_rate']
            * get_month_multiplier(month)
            * (1 + np.random.normal(0, 0.08)),
            2
        )

        if is_booked:

            stay_length = random.randint(2, 7)
            checkout_date = current_date + timedelta(days=stay_length)

            # Avoid overrun
            if checkout_date > pd.to_datetime(END_DATE):
                break

            booking_id = f'BK{booking_counter:06}'

            gross_revenue = round(nightly_rate * stay_length, 2)
            cleaning_fee = random.choice([50, 60, 70, 80, 90])
            platform_fee = round(gross_revenue * 0.03, 2)
            management_fee = round(gross_revenue * 0.20, 2)
            maintenance_cost = round(random.uniform(10, 60), 2)
            total_cost = cleaning_fee + platform_fee + maintenance_cost
            profit = gross_revenue - total_cost

            booking_platform = random.choices(
                platforms,
                weights=[0.65, 0.25, 0.10]
            )[0]

            guest_rating = round(
                min(max(np.random.normal(4.7, 0.25), 3.5), 5.0),
                1
            )

            bookings.append({
                'booking_id': booking_id,
                'property_id': prop['property_id'],
                'platform': booking_platform,
                'guest_name': fake.name(),
                'booking_date': current_date - timedelta(days=random.randint(5, 60)),
                'checkin_date': current_date,
                'checkout_date': checkout_date,
                'stay_nights': stay_length,
                'gross_booking_revenue': gross_revenue,
                'cleaning_fee': cleaning_fee,
                'platform_fee': platform_fee,
                'management_fee': management_fee,
                'maintenance_cost': maintenance_cost,
                'total_cost': total_cost,
                'profit': round(profit, 2),
                'avg_nightly_rate': nightly_rate,
                'review_rating': guest_rating,
                'occupancy_status': 'Completed'
            })

            # Create daily occupancy records
            for stay_day in pd.date_range(current_date, checkout_date - timedelta(days=1)):

                booked_dates.add(stay_day)

                property_daily.append({
                    'property_id': prop['property_id'],
                    'calendar_date': stay_day,
                    'day_of_week': stay_day.day_name(),
                    'month_name': stay_day.month_name(),
                    'year': stay_day.year,
                    'available_flag': 1,
                    'booked_flag': 1,
                    'nightly_rate': nightly_rate,
                    'daily_revenue': round(gross_revenue / stay_length, 2),
                    'booking_id': booking_id
                })

            # Guest review table
            reviews.append({
                'review_id': f'RV{review_counter:06}',
                'booking_id': booking_id,
                'property_id': prop['property_id'],
                'review_date': checkout_date + timedelta(days=random.randint(1, 5)),
                'guest_rating': guest_rating,
                'review_text': fake.sentence(nb_words=12)
            })

            booking_counter += 1
            review_counter += 1

            current_date = checkout_date

        else:

            property_daily.append({
                'property_id': prop['property_id'],
                'calendar_date': current_date,
                'day_of_week': current_date.day_name(),
                'month_name': current_date.month_name(),
                'year': current_date.year,
                'available_flag': 1,
                'booked_flag': 0,
                'nightly_rate': nightly_rate,
                'daily_revenue': 0,
                'booking_id': None
            })

            current_date += timedelta(days=1)

# =========================================================
# CREATE DATAFRAMES
# =========================================================

bookings_df = pd.DataFrame(bookings)
property_daily_df = pd.DataFrame(property_daily)
reviews_df = pd.DataFrame(reviews)

# =========================================================
# INTENTIONALLY MESSY DATA ISSUES
# =========================================================

# Duplicate rows
if len(bookings_df) > 5:
    duplicate_rows = bookings_df.sample(5)
    bookings_df = pd.concat([bookings_df, duplicate_rows], ignore_index=True)

# Missing values
for col in ['review_rating', 'platform_fee']:
    bookings_df.loc[
        bookings_df.sample(frac=0.01).index,
        col
    ] = np.nan

# Inconsistent platform casing
bookings_df.loc[
    bookings_df.sample(frac=0.02).index,
    'platform'
] = 'airbnb'

# =========================================================
# EXPORT CSV FILES
# =========================================================

bookings_df.to_csv(f'{OUTPUT_FOLDER}/raw_bookings.csv', index=False)
property_daily_df.to_csv(f'{OUTPUT_FOLDER}/raw_property_daily.csv', index=False)
reviews_df.to_csv(f'{OUTPUT_FOLDER}/raw_reviews.csv', index=False)

end = perf_counter()
# =========================================================
# SUMMARY
# =========================================================

print('Synthetic Airbnb management raw data created successfully!')
print()
print(f'Properties: {len(properties_df):,}')
print(f'Bookings: {len(bookings_df):,}')
print(f'Daily Property Records: {len(property_daily_df):,}')
print(f'Reviews: {len(reviews_df):,}')
print()
print(f'Files saved in: {OUTPUT_FOLDER}/')
print(f"Elapsed time: {end - start:.6f} seconds")

Synthetic Airbnb management raw data created successfully!

Properties: 50
Bookings: 11,434
Daily Property Records: 54,650
Reviews: 11,429

Files saved in: C:\Users\geoff\Downloads\Key Docs\7. Airbnb_dbt_snowflake_project\AIRBNB_DBT_PROJECT\source_data/
Elapsed time: 39.016218 seconds
